# 2일차 · 코드 없이 테스트 돌려 보기

이 노트북은 **코드를 읽거나 고칠 필요가 없습니다.** 각 칸의 ▶ 버튼을 누르고, 오른쪽 입력 칸만 바꾸면 됩니다.

> 우리는 PM으로서 **"무엇을 확인해야 하는가"를 정하고, 결과를 읽고, 판단하는 사람**입니다. 테스트 코드는 개발자가 쓰고, 테스트 **케이스**는 PM·기획자도 씁니다.

| 순서 | 교시 | 하는 일 |
| --- | --- | --- |
| 0 | 1 | 준비 (▶ 한 번) |
| 1 | 1 | 자동 테스트 전체 실행, 결과표 읽기 |
| 2 | 2 | 서비스 직접 써 보기 → 테스트 케이스 표에 행 추가 → 결함 찾기 |
| 3 | 3 | 손 계산 vs 도구 결과 비교 → 결함 찾기, 품질 게이트 기준 정하기 |
| 4 | 5 | 성능 리포트 한 번에 만들기, 숫자 검토 |

처음 ▶를 누르면 "이 노트북은 Google에서 작성하지 않았습니다" 경고가 뜹니다. **무시하고 실행**을 누르세요.

In [ ]:
#@title 0. 준비 (▶ 누르고 30초 기다리기)
import os, sys
if not os.path.exists("/content/ai-quality-lab"):
    !git clone -q https://github.com/leejaehee1/ai-quality-lab.git /content/ai-quality-lab
%cd /content/ai-quality-lab/day2
!pip install -q pytest
sys.path.insert(0, "/content/ai-quality-lab/day2/notebooks")
from lab_helpers import *
from IPython.display import display, Markdown, Image
print("준비 완료")

## 1. 자동 테스트 전체 실행 (1교시)

개발자가 미리 만들어 둔 자동 테스트가 수십 개 있습니다. 버튼 한 번으로 모두 실행됩니다.

**읽는 법**: 영역(무엇을 검사하나) → 확인하는 것 → 결과. 지금은 모두 ✅입니다.

생각해 볼 질문: **"테스트가 모두 통과했다 = 결함이 없다"일까요?** (1일차 테스팅 원칙 1번)

In [ ]:
#@title 1. 전체 테스트 실행
display(highlight(run_tests()))

## 2. 테스트 케이스 표로 결함 찾기 (2교시)

### 2-1. 요구사항

> 청년 지원금 대상: **만 19세 이상 34세 이하**, 해당 지역 **거주 6개월 이상**, **미취업** 상태

### 2-2. 서비스 직접 써 보기 (탐색적 테스트)

슬라이더와 목록을 바꾸고 ▶를 누르세요. 요구사항과 다른 판정이 나오는 입력을 찾아보세요.
힌트: 1일차에 배운 **경계값**(18, 19, 34, 35 / 5, 6개월)을 넣어 보세요.

In [ ]:
#@title 2-2. 판정 서비스 써 보기
나이 = 19 #@param {type:"slider", min:0, max:120, step:1}
거주_개월 = 6 #@param {type:"slider", min:0, max:60, step:1}
취업 = "아니오" #@param ["아니오", "예"]
try_service(나이, 거주_개월, 취업)

### 2-3. 테스트 케이스 표

이 표의 **한 행이 자동 테스트 하나**입니다. 개발자는 이 표를 읽어 매번 자동으로 확인합니다.
지금은 4개 행만 있고, 모두 통과합니다.

In [ ]:
#@title 2-3. 현재 테스트 케이스 표 보기
display(show_cases())

### 2-4. 행 추가하기

경계값을 중심으로 케이스를 추가하세요. **기대 결과는 요구사항 기준**으로 적습니다(서비스가 주는 답이 아니라).
한 행 추가할 때마다 ▶를 누릅니다. 잘못 넣었으면 2-6으로 마지막 행을 지웁니다.

In [ ]:
#@title 2-4. 테스트 케이스 추가
나이 = 18 #@param {type:"slider", min:0, max:120, step:1}
거주_개월 = 12 #@param {type:"slider", min:0, max:60, step:1}
취업 = "아니오" #@param ["아니오", "예"]
기대_결과 = "비대상" #@param ["대상", "비대상"]
이유 = "나이 하한 경계값 바로 아래" #@param {type:"string"}
add_case(나이, 거주_개월, 취업, 기대_결과, 이유)
display(show_cases())

In [ ]:
#@title 2-5. 테스트 케이스 표만 실행
display(highlight(run_tests(["tests/test_case_table.py"])))

In [ ]:
#@title 2-6. (필요할 때만) 마지막 행 지우기
remove_last_case()
display(show_cases())

❌가 나왔다면 **결함을 찾은 것**입니다. 실패 이유 칸의 "기대 / 실제"를 그대로 버그 리포트에 옮겨 적으면 됩니다.

팀 토론: 처음 4개 행으로는 왜 이 결함을 못 찾았을까요? 어떤 행이 **꼭 있어야** 했나요?

## 3. 손 계산 vs 도구 결과 (3교시)

AI 서비스는 "정답"을 미리 알기 어렵습니다. 그래서 **작은 예제를 손으로 계산해 도구와 비교**하는 방법을 씁니다.

### 3-1. 빈칸 채우기 (결측값)

학습 데이터와 테스트 데이터에 나이가 빈칸(NaN)인 사람이 있습니다. 빈칸은 **학습 데이터의 중앙값**으로 채우기로 했습니다.
(1일차 5교시: 테스트 데이터 정보를 쓰면 **데이터 누수**)

In [ ]:
#@title 3-1. 예제 데이터 보기
display(Markdown("**학습 데이터**")); display(MEDIAN_TRAIN)
display(Markdown("**테스트 데이터**")); display(MEDIAN_TEST)

In [ ]:
#@title 3-1. 내 손 계산 입력 → 도구와 비교
학습_중앙값 = 0 #@param {type:"number"}
check_median(학습_중앙값)

### 3-2. 정밀도 (분류 지표)

암 진단 AI입니다. 우리가 찾고 싶은 것은 **악성**입니다(양성 클래스 = 악성).
정밀도 = AI가 "악성"이라고 한 것 중 **실제 악성**의 비율.

In [ ]:
#@title 3-2. 예측 결과 보기
display(PRECISION_EXAMPLE)

In [ ]:
#@title 3-2. 내 손 계산 입력 (소수, 예: 0.75) → 도구와 비교
악성_정밀도 = 0.0 #@param {type:"number"}
check_precision(악성_정밀도)

### 3-3. 품질 게이트 정하기

**품질 게이트** = "이 기준을 못 넘으면 배포하지 않는다"는 팀의 약속. PM이 정해야 하는 값입니다.

- 교차 검증 재현율: 악성 환자를 놓치지 않는 비율의 **최소 기준**
- 잡음 하락 허용치: 입력이 조금 흔들릴 때 재현율이 **떨어져도 되는 최대치**

슬라이더로 기준을 바꿔 보며, 어디서부터 ❌(배포 불가)가 되는지 확인하세요.

In [ ]:
#@title 3-3. 품질 게이트 실행
최소_재현율 = 0.9 #@param {type:"slider", min:0.80, max:1.00, step:0.01}
잡음_하락_허용치 = 0.02 #@param {type:"slider", min:0.00, max:0.10, step:0.01}
r = cv_recall()
print(f"현재 모델의 교차 검증 재현율: {r['mean']:.3f} ± {r['std']:.3f}  (폴드별 {r['folds']})")
display(highlight(run_tests(["tests/test_model.py"],
                            env={"MIN_CV_RECALL": 최소_재현율, "MAX_NOISE_DROP": 잡음_하락_허용치})))

팀 토론: 암 진단 서비스라면 최소 재현율을 얼마로 하시겠어요? 쇼핑 추천 서비스라면요? 근거를 한 문장으로 적어 보세요.

## 4. 성능 리포트 자동 생성 (5교시)

버튼 한 번으로 리포트(표 + 혼동 행렬 그림)가 만들어집니다. CI에서도 같은 리포트가 매번 자동으로 만들어집니다.

**검토 포인트**: 혼동 행렬 그림의 숫자로 정밀도를 직접 계산해, 표의 precision과 **같은지** 확인하세요.

In [ ]:
#@title 4. 성능 리포트 만들기
from mlqa.report import generate_report
generate_report("reports")
display(Markdown(open("reports/model_report.md", encoding="utf-8").read()))
display(Image("reports/confusion_matrix.png", width=380))

리포트를 읽은 뒤 [리포트 개선안 양식](https://github.com/leejaehee1/ai-quality-lab/blob/main/day2/templates/report_redesign.md)으로 "PM이 보고 싶은 리포트"를 설계합니다.